# Doplnenie seedov pre n = 100 (Colab)

Spustí **našu implementáciu** aj **pôvodný C++ program autorov** na všetkých 360 inštanciách
s n = 100 pre zvolené seedy a výsledky ukladá priebežne na **Google Drive**
(prerušený beh sa po opätovnom spustení dokončí – hotové behy sa preskočia).

Postup: nastav `SEEDS` v bunke nižšie → **Runtime → Run all** → povoľ prístup k Drive.
Podrobnosti a odhad času: `colab/README.md` v repozitári.

In [16]:
# ======== NASTAVENIA ========
SEEDS = [10]             # seedy pre túto session (4,5 .. 6,7 .. 8,9 .. 10
RUN_OURS = True            # naša implementácia (Python + numba)
RUN_AUTHORS = True         # pôvodný program autorov (C++)
BRANCH = "etapa-2-faza2-npsset"
DRIVE_DIR = "/content/drive/MyDrive/npfs-mo-colab"
# ============================
TAG = "s" + "-".join(map(str, SEEDS))
print("Session", TAG)

Session s10


In [17]:

from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Výsledky:", DRIVE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Výsledky: /content/drive/MyDrive/npfs-mo-colab


In [18]:
%%bash -s "$BRANCH"
# 2) Náš repozitár a Python balíky
set -e
cd /content
rm -rf replication-npfs-mo
git clone -q --depth 1 -b "$1" https://github.com/svihuralukas/replication-npfs-mo.git
cd replication-npfs-mo
git log --oneline -1
pip -q install -r requirements.txt
python --version

0bf5c65 Delete colab/n100_seedy.ipynb
Python 3.13.16


In [19]:
%%bash
# 3) Pôvodný program autorov – rovnaký commit ako v našich referenčných behoch.
# CMake projektu chce Boost >= 1.75, Colab má 1.74, preto kompilujeme priamo cez g++
# (overené: výsledky sú bit po bite rovnaké ako z buildu cez CMake).
set -e
apt-get -qq update > /dev/null
DEBIAN_FRONTEND=noninteractive apt-get -qq install -y libboost-program-options-dev libboost-timer-dev \
    libboost-system-dev libboost-chrono-dev libboost-filesystem-dev > /dev/null
cd /content
rm -rf npfs-mo
git clone -q https://github.com/mrpritt/npfs-mo.git
cd npfs-mo
git checkout -q bde1bc3
git submodule -q update --init
cd src
g++ -O3 -DNDEBUG -std=c++17 -DIL_STD -fexceptions -Ifmt/include \
    npfsmo.cpp instance.cpp solution.cpp heuristics.cpp holes.cpp options.cpp random.cpp logging.cpp helpers.cpp \
    -o /content/npfsmo -lboost_program_options -lboost_timer -lboost_system -lboost_chrono -lboost_filesystem -lrt -ldl
/content/npfsmo --flowtime --timelimit -1 --iterfactor 0.1 --npfs /content/replication-npfs-mo/tests/fig3.txt | grep INFO
echo "(posledné ΣC v riadku INFO musí byť 1191)"

INFO fig3.txt 410 1216 0.000496204 410 1216 0.000496053 410 1194 0.000541102 410 1216 0.000496053 410 1194 0.000552631 410 1216 0.000496053 410 1191 0.020409085 
(posledné ΣC v riadku INFO musí byť 1191)


shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/dists/noble/InRelease  Connection failed [IP: 185.125.189.186 443]
W: Failed to fetch https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu/dists/noble/InRelease  Could not wait for server fd - select (11: Resource temporarily unavailable) [IP: 185.125.189.187 443]
W: Some index files failed to download. They have been ignored, or old ones used instead.
chdir: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory


In [20]:
# 4) Kontrola prostredia: testy a zhoda so seedom 2 z našich referenčných behov
#    (naša implementácia aj program autorov musia dať presne rovnaké ΣC ako na notebooku)
import os, subprocess, csv, io
os.chdir("/content/replication-npfs-mo")
r = subprocess.run(["python", "-m", "pytest", "tests/test_nonperm.py", "tests/test_nps_set.py", "-q"],
                   capture_output=True, text=True)
print(r.stdout.strip().splitlines()[-1])
assert r.returncode == 0, r.stdout

CHECK = ["0.3_20_10_00", "0.4_50_20_01"]
ref_o = {(r["instance"]): r for r in csv.DictReader(open("results/iga_ours.csv")) if r["seed"] == "2"}
ref_a = {(r["instance"]): r for r in csv.DictReader(open("reference/iga_authors.csv")) if r["seed"] == "2"}
tmp = "/content/check"
os.makedirs(tmp, exist_ok=True)
files = [f"instances/generated/{i}.txt" for i in CHECK]
subprocess.run(["python", "src/run_experiments.py", "--instances", *files, "--seeds", "2",
                "--out", f"{tmp}/o.csv"], check=True, capture_output=True)
subprocess.run(["python", "reference/run_authors.py", "--npfsmo", "/content/npfsmo", "--instances", *files,
                "--seeds", "2", "--out", f"{tmp}/a.csv"], check=True, capture_output=True)
ok = True
for name, out, ref in (("naša", f"{tmp}/o.csv", ref_o), ("autori", f"{tmp}/a.csv", ref_a)):
    for row in csv.DictReader(open(out)):
        same = all(row[k] == ref[row["instance"]][k] for k in ("iga0_ft", "nps_ft", "iga_ft"))
        ok &= same
        print(f"{name:7s} {row['instance']}: IGA ΣC {row['iga_ft']} vs {ref[row['instance']]['iga_ft']}",
              "OK" if same else "NEZHODA")
assert ok, "Výsledky sa nezhodujú s referenciou – NEPOKRAČOVAŤ, napíš mi."
print("Prostredie je v poriadku.")

57 passed in 15.44s
naša    0.3_20_10_00: IGA ΣC 14203 vs 14203 OK
naša    0.4_50_20_01: IGA ΣC 76072 vs 76072 OK
autori  0.3_20_10_00: IGA ΣC 14203 vs 14203 OK
autori  0.4_50_20_01: IGA ΣC 75537 vs 75537 OK
Prostredie je v poriadku.


In [21]:
# 5) Hlavný beh: pre každý seed najprv naša implementácia, potom program autorov.
#    Zapisuje sa priebežne na Drive; po odpojení stačí znova Run all (hotové sa preskočí).
import os, subprocess, platform, datetime
os.chdir("/content/replication-npfs-mo")
jobs = os.cpu_count()
cpu = subprocess.run("lscpu | grep 'Model name' | cut -d: -f2", shell=True, capture_output=True, text=True).stdout.strip()
with open(f"{DRIVE_DIR}/info_{TAG}.txt", "a") as f:
    f.write(f"{datetime.datetime.now().isoformat()} start {TAG} cpu='{cpu}' cores={jobs} python={platform.python_version()}\n")
print("CPU:", cpu, "| jadrá:", jobs)

out_o = f"{DRIVE_DIR}/iga_ours_colab_{TAG}.csv"
hist_o = f"{DRIVE_DIR}/iga_ours_colab_{TAG}_history.csv"
out_a = f"{DRIVE_DIR}/iga_authors_colab_{TAG}.csv"
for s in SEEDS:
    if RUN_OURS:
        print(f"\n=== seed {s}: naša implementácia ===", flush=True)
        subprocess.run(["python", "src/run_experiments.py", "--instances", "instances/generated/*.txt",
                        "--min-n", "100", "--seeds", str(s), "--jobs", str(jobs),
                        "--out", out_o, "--history", hist_o], check=True)
    if RUN_AUTHORS:
        print(f"\n=== seed {s}: program autorov ===", flush=True)
        subprocess.run(["python", "reference/run_authors.py", "--npfsmo", "/content/npfsmo",
                        "--instances", "instances/generated/*.txt", "--min-n", "100", "--seeds", str(s),
                        "--jobs", str(jobs), "--out", out_a], check=True)
with open(f"{DRIVE_DIR}/info_{TAG}.txt", "a") as f:
    f.write(f"{datetime.datetime.now().isoformat()} done {TAG}\n")
print("\nHOTOVO")

CPU: Intel(R) Xeon(R) CPU @ 2.20GHz | jadrá: 12

=== seed 10: naša implementácia ===

=== seed 10: program autorov ===

HOTOVO


In [22]:
# 6) Súhrn – koľko behov je hotových (každý seed má mať 360 riadkov v oboch súboroch)
import pandas as pd, os
for f in sorted(os.listdir(DRIVE_DIR)):
    if f.endswith(".csv") and "history" not in f:
        d = pd.read_csv(f"{DRIVE_DIR}/{f}")
        print(f, d.groupby("seed").size().to_dict())

iga_authors_colab_s10.csv {10: 360}
iga_authors_colab_s4-5.csv {4: 360, 5: 360}
iga_authors_colab_s6-7.csv {6: 360, 7: 360}
iga_authors_colab_s8-9.csv {8: 360, 9: 360}
iga_ours_colab_s10.csv {10: 360}
iga_ours_colab_s4-5.csv {4: 360, 5: 360}
iga_ours_colab_s6-7.csv {6: 360, 7: 360}
iga_ours_colab_s8-9.csv {8: 360, 9: 360}
